# Bonsai 27B Q1: Colab T4 + prebuilt CUDA server + secure ngrok API

This is the final clean notebook. It uses a **prebuilt CUDA wheel**: no `git clone`, no CMake, no NVCC, and no llama.cpp source build. It also always kills a previous local server on port 8080 before starting exactly one new server.

Prerequisites: select a GPU runtime in Colab, have the model in Google Drive at the configured path, and create a free ngrok authtoken.


In [ ]:
!nvidia-smi
!python --version


In [ ]:
# Install binary packages only. --only-binary prevents a hidden source compile fallback.
%pip -q uninstall -y llama-cpp-python
%pip -q install --only-binary=:all: --upgrade --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 'llama-cpp-python[server]' pyngrok httpx


In [ ]:
# Fail early if pip installed a CPU-only package.
from llama_cpp import llama_supports_gpu_offload
assert llama_supports_gpu_offload(), 'CUDA GPU offload is unavailable. Restart with a GPU runtime, then rerun.'
print('Prebuilt CUDA llama-cpp-python wheel verified.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_MODEL_PATH = Path('/content/drive/MyDrive/models/Bonsai-27B-Q1_0.gguf')
LOCAL_MODEL_PATH = Path('/content/Bonsai-27B-Q1_0.gguf')

if not DRIVE_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Model not found: {DRIVE_MODEL_PATH}')

# Always serve from Colab SSD, never directly from mounted Drive.
if not LOCAL_MODEL_PATH.is_file() or LOCAL_MODEL_PATH.stat().st_size != DRIVE_MODEL_PATH.stat().st_size:
    !rsync -ah --info=progress2 "$DRIVE_MODEL_PATH" "$LOCAL_MODEL_PATH"

print('Using:', LOCAL_MODEL_PATH)
print('Size GiB:', round(LOCAL_MODEL_PATH.stat().st_size / 1024**3, 2))


In [ ]:
from getpass import getpass

NGROK_AUTHTOKEN = getpass('Paste NGROK_AUTHTOKEN (hidden): ').strip()
API_KEY = getpass('Create/paste a strong API key, at least 20 characters (hidden): ').strip()

if not NGROK_AUTHTOKEN:
    raise ValueError('NGROK_AUTHTOKEN is required.')
if len(API_KEY) < 20:
    raise ValueError('API_KEY must contain at least 20 characters.')


In [ ]:
# CLEAN RESET: stop only old processes listening on the chosen local API port.
# This fixes: [Errno 98] address already in use.
import subprocess, time, socket

PORT = 8080
subprocess.run(['bash', '-lc', f'fuser -k {PORT}/tcp 2>/dev/null || true'], check=False)
time.sleep(4)

def port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(('127.0.0.1', port)) != 0

if not port_is_free(PORT):
    listeners = subprocess.run(
        ['bash', '-lc', f'lsof -nP -iTCP:{PORT} -sTCP:LISTEN || true'],
        capture_output=True, text=True, check=False
)
    raise RuntimeError(f'Port {PORT} is still occupied:\n{listeners.stdout}')

print(f'Port {PORT} is free.')


In [ ]:
# Start exactly ONE prebuilt CUDA-backed OpenAI-compatible server.
import subprocess, time, httpx

CONTEXT_SIZE = 8192
LOG_PATH = '/content/bonsai-server.log'

command = [
    'python', '-m', 'llama_cpp.server',
    '--model', str(LOCAL_MODEL_PATH),
    '--n_gpu_layers', '-1',
    '--n_ctx', str(CONTEXT_SIZE),
    '--n_batch', '512',
    '--host', '127.0.0.1',
    '--port', str(PORT),
    '--api_key', API_KEY,
]

with open(LOG_PATH, 'w') as log_file:
    server_process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT)

for _ in range(180):
    if server_process.poll() is not None:
        print(Path(LOG_PATH).read_text(errors='replace')[-8000:])
        raise RuntimeError('Model server exited during startup.')
    try:
        response = httpx.get(f'http://127.0.0.1:{PORT}/docs', timeout=3)
        if response.status_code == 200:
            print('Server ready on http://127.0.0.1:8080')
            break
    except httpx.HTTPError:
        pass
    time.sleep(2)
else:
    print(Path(LOG_PATH).read_text(errors='replace')[-8000:])
    raise TimeoutError('Timed out waiting for server startup.')


In [ ]:
# Confirm that the loaded model uses T4 VRAM.
!nvidia-smi
!tail -n 80 /content/bonsai-server.log


In [ ]:
# Local authenticated OpenAI-compatible test.
import json
HEADERS = {'Authorization': f'Bearer {API_KEY}'}

models_response = httpx.get(f'http://127.0.0.1:{PORT}/v1/models', headers=HEADERS, timeout=30)
models_response.raise_for_status()
models = models_response.json()
MODEL_ID = models['data'][0]['id']
print('MODEL_ID:', MODEL_ID)

payload = {
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: Bonsai API is online.'}],
    'temperature': 0,
    'max_tokens': 32,
    'stream': False,
}

local_response = httpx.post(
    f'http://127.0.0.1:{PORT}/v1/chat/completions',
    headers=HEADERS, json=payload, timeout=300
)
local_response.raise_for_status()
local_result = local_response.json()
print(local_result['choices'][0]['message']['content'])
print(json.dumps(local_result.get('usage', {}), indent=2))


In [ ]:
# Publish only the API-key-protected server through a temporary HTTPS ngrok endpoint.
from pyngrok import ngrok

try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(PORT, 'http')
PUBLIC_URL = tunnel.public_url.rstrip('/')
print('Public API base URL:', PUBLIC_URL + '/v1')


In [ ]:
# End-to-end external route verification.
public_response = httpx.post(
    f'{PUBLIC_URL}/v1/chat/completions',
    headers=HEADERS, json=payload, timeout=300
)
public_response.raise_for_status()
print(public_response.json()['choices'][0]['message']['content'])
print('ngrok API verified.')


In [ ]:
print(f'''export OPENAI_BASE_URL="{PUBLIC_URL}/v1"
export OPENAI_API_KEY="YOUR_API_KEY"

curl -sS "$OPENAI_BASE_URL/chat/completions" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{{
    "model": "{MODEL_ID}",
    "messages": [{{"role": "user", "content": "Reply with exactly: Remote API works."}}],
    "temperature": 0,
    "max_tokens": 24
  }}' | jq
''')


In [ ]:
# Shutdown when finished.
try:
    ngrok.disconnect(PUBLIC_URL)
    ngrok.kill()
except Exception:
    pass

if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()

print('Server and tunnel stopped.')
